In [20]:
# imports

import os
import gradio as gr
import json

from openai import OpenAI
from dotenv import load_dotenv

In [21]:
# load environment variables
load_dotenv(override=True)

# get the API key from the environment variable
openai_api_key = os.getenv("OPENAI_API_KEY")

# initialize OpenAI client
client = OpenAI(api_key=openai_api_key)

In [32]:
MODEL = "gpt-5.6-luna"

In [23]:
SYSTEM_PROMPT = """
You are a helpful assistant for Airline called FlightAI.
Give short, courteous answers, not more than 1 sentence.
Always be accurate, If you don't know the answer say so, don't hallucinate.
"""

In [24]:
import sqlite3

DB_NAME = 'ticket_prices.db'

def get_connection():
    """Creates a connection with a busy timeout to avoid 'database is locked' errors."""
    return sqlite3.connect(DB_NAME, timeout=10)

def create_table():
    with get_connection() as conn:
        cursor = conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS prices (
                city TEXT PRIMARY KEY,
                price REAL
            )
        ''')
        conn.commit()

def insert_prices(ticket_data):
    """
    Inserts one or more employee rows.
    Skips duplicates based on the UNIQUE constraint on 'name'.
    Accepts a single tuple or a list of tuples.
    """
    if isinstance(ticket_data, tuple):
        ticket_data = [ticket_data]

    with get_connection() as conn:
        cursor = conn.cursor()

        cursor.execute("SELECT COUNT(*) FROM prices")
        before = cursor.fetchone()[0]

        cursor.executemany('''
            INSERT OR IGNORE INTO prices (city, price)
            VALUES (?, ?)
        ''', ticket_data)
        conn.commit()

        cursor.execute("SELECT COUNT(*) FROM prices")
        after = cursor.fetchone()[0]
        
        inserted = after - before
        skipped = len(ticket_data) - inserted
        
        print(f"Inserted {inserted} new row(s). Skipped {skipped} duplicate(s).")


# --- Usage ---
create_table()

insert_prices([  # bulk insert
    ('london', 8500),  # duplicate — skipped
    ('new york', 6200),
    ('berlin', 9100),
    ('singapore', 5800),
    ('dublin', 4500),
    ('madrid', 3769)
])

Inserted 0 new row(s). Skipped 6 duplicate(s).


In [25]:

def get_ticket_prices(destination_city):
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        print(destination_city.lower())
        cursor.execute("SELECT price FROM prices WHERE city = ?", (destination_city.lower(),))
        result = cursor.fetchone()
    return f"The price of a ticket to {destination_city} is ${int(result[0])}" if result else f"No price data available for this {destination_city}"

In [27]:
price_function = {
    "name": "get_ticket_prices",
    "description": "Get the price of a ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to"
            }
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [28]:
tools = [
    {
        "type": "function",
        "function": price_function
    }
]

In [29]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_prices":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            print(city)
            price_details = get_ticket_prices(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })

    return responses    


In [30]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(messages=messages, model=MODEL, tools=tools, reasoning_effort="none")

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = client.chat.completions.create(messages=messages, model=MODEL, tools=tools, reasoning_effort="none")

    return response.choices[0].message.content

In [31]:
gr.ChatInterface(fn=chat, title="Airline AI Assistant", description="Ask anything you want").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Madrid
madrid
